# Architecture 1 — one-seed training (research record §40 step 6, §46)

Run AFTER `stage4_v2_aptos_cache.ipynb` (bundle exists) and the laptop C2 screen. GPU runtime. **Cell 5 trains.**
P protocol reused exactly (batch 2, AdamW 1e-4 / wd 0.05, weighted CORN, ≤ 50 epochs, val-QWK early stopping); seed 42 only.
Refuses to run on a missing/mismatched model SHA, an incomplete or legacy cache, failed Stage-3/RGB parity, an inconsistent
bundle fingerprint, the wrong APTOS split, or non-APTOS (e.g. IDRiD) ids.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import stage4_v2_setup
SETUP_INFO = stage4_v2_setup.setup_stage4_v2(fetch_weights=False)
import glob, json, posixpath
import numpy as np, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_model as a1, arch1_train as at
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: Stage-4 model identity, bundle, protocol, C2 decision ====
EXP = colab_config.DRIVE.experiments_root
STAGE4_RUN_DIR = posixpath.join(EXP, "Stage4V2", "stage4v2_k4_seed42_bs8")
MODEL_SHA = json.load(open(posixpath.join(STAGE4_RUN_DIR, "exported_model.json")))["model_sha256"]
GEN4 = cache.stage4_generation_id(MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
C2_RESULTS = None                   # set to the Drive path of the laptop C2 results.json (experiments/C2/<run>/results.json)
ACKNOWLEDGE_C2_FAIL = False         # §40: if C2 FAILED, Architecture 1 is only a clearly low-prior test
assert C2_RESULTS and os.path.exists(C2_RESULTS), "run the C2 screen first and set C2_RESULTS"
C2 = json.load(open(C2_RESULTS))
assert C2["decision"]["PASS"] or ACKNOWLEDGE_C2_FAIL, "C2 FAILED -- set ACKNOWLEDGE_C2_FAIL only for a low-prior test"
SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
CLASS_WEIGHTS = list(PREREG["class_weights"])
at.assert_protocol(PREREG, CLASS_WEIGHTS)
SEED = at.FIRST_SEED
RUN_DIR = posixpath.join(EXP, "Architecture1", f"arch1_{MODEL_SHA[:12]}_seed{SEED}")
print("model", MODEL_SHA, "| bundle", BUNDLE_ID, "| C2", C2["decision"]["PASS"], "| run", RUN_DIR)

In [ ]:
# ==== [3] VERIFY THE BUNDLE ON DRIVE, STAGE IT LOCALLY, VERIFY AGAIN ====
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=MODEL_SHA)
stage4_v2_setup.stage_bundle(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=MODEL_SHA, roots=LOCAL_ROOTS)
BUNDLE.require(BUNDLE.population)
print(json.dumps(BUNDLE.describe(), indent=1))

In [ ]:
# ==== [4] PINNED CONVNEXT WEIGHTS + PRE-FLIGHT (zero-initialised model == P on real samples) ====
WEIGHTS = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))
REFERENCE, REFERENCE_ARRAYS = pl.load_reference(WEIGHTS["local_path"])
model = at.build_compiled_model(BUNDLE.channels, SEED, REFERENCE, CLASS_WEIGHTS, mixed_precision=False)
p_model = pl.build_pl_model("P", SEED, REFERENCE_ARRAYS)
samples = [BUNDLE.load_sample(i) for i in BUNDLE.val_ids[:2]]
x = {k: np.stack([s[k] for s in samples]) for k in ("rgb", "vessel", "pathology")}
s5 = np.concatenate([x["rgb"], np.zeros(x["rgb"].shape[:3] + (5,), "float32")], -1)
p_logits = p_model.predict([s5, np.zeros((2, 256, 256, 3), "float32"), np.zeros((2, 1), "float32")], verbose=0)
diff = float(np.max(np.abs(model.predict_on_batch(x) - p_logits)))
assert diff <= pl.PL_P_EQUIVALENCE_TOL, f"zero-initialised Architecture 1 != P ({diff})"
print("pre-flight OK: |Arch1_init - P| =", diff, "| parameters", a1.parameter_report(model))
del model, p_model

In [ ]:
# ==== [5] TRAIN SEED 42 (resumable) -- THIS TRAINS ====
at.train_seed(RUN_DIR, BUNDLE, SEED, REFERENCE, CLASS_WEIGHTS, repo_dir=REPO_DIR, staging_dir="/content/arch1_staging")

In [ ]:
# ==== [6] EVALUATE BEST + LAST, permutation contributions, pre-specified one-seed checks vs P-42 ====
P42 = sorted(glob.glob(posixpath.join(EXP, "PL_ConvNeXtPriors", "*", "P_convnext_rgb", "seed_42", "evaluation", "metrics_best.json")))
assert len(P42) == 1, P42
RES = at.evaluate_run(RUN_DIR, BUNDLE, SEED, REFERENCE, CLASS_WEIGHTS, p42_metrics=json.load(open(P42[0])))
print(json.dumps({k: {"qwk": v["metrics"]["qwk"], "auroc_ge3_g4_vs_g012": v["metrics"]["auroc_ge3_g4_vs_g012"],
                      "permutation": v["permutation"]} for k, v in RES.items()}, indent=1))
print(json.dumps(RES["best"]["one_seed_checks"], indent=1))